# NLP Pipeline and Applications

Individual NLP techniques — tokenization, TF-IDF, embeddings, classifiers, sequence models, language models — are building blocks. Production systems combine them into **pipelines** that ingest raw text and deliver useful outputs: classified documents, extracted entities, generated summaries, or conversational responses.

This notebook ties the field together. It walks through end-to-end pipeline design, demonstrates a complete classical NLP workflow in code, surveys major real-world applications, and covers the practical decisions — model selection, evaluation, deployment, and responsible use — that practitioners face when shipping NLP systems.

In [ ]:
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. The End-to-End NLP Pipeline

Every NLP system, regardless of complexity, follows the same high-level flow:

```
┌─────────────┐    ┌──────────────┐    ┌─────────────┐    ┌──────────────┐
│  Data       │ →  │ Preprocess   │ →  │ Represent   │ →  │ Model        │
│  Collection │    │ & Tokenize   │    │ (features)  │    │ (predict)    │
└─────────────┘    └──────────────┘    └─────────────┘    └──────────────┘
                                                                  ↓
┌─────────────┐    ┌──────────────┐    ┌─────────────┐    ┌──────────────┐
│  Monitor &  │ ←  │ Deploy       │ ←  │ Evaluate    │ ←  │ Post-process │
│  Iterate    │    │              │    │             │    │              │
└─────────────┘    └──────────────┘    └─────────────┘    └──────────────┘
```

**Data collection** — gather text and labels from databases, APIs, web scraping, or annotation platforms.

**Preprocessing** — clean, normalize, tokenize. Choices depend on the task and model.

**Representation** — convert tokens to features (TF-IDF, embeddings) or feed directly to a neural model.

**Modeling** — train a classifier, sequence tagger, or language model.

**Post-processing** — format outputs, apply business rules, filter low-confidence predictions.

**Evaluation** — measure performance on held-out data with task-appropriate metrics.

**Deployment** — serve the model via API, batch job, or embedded in an application.

**Monitoring** — track accuracy, latency, and data drift in production; retrain when performance degrades.

---

## 2. Classical vs Neural vs LLM Pipelines

| Stage | Classical | Neural | LLM |
|-------|-----------|--------|-----|
| **Preprocessing** | Heavy (stemming, stop words) | Light (subword tokenization) | Minimal (tokenizer only) |
| **Features** | TF-IDF, BoW | Learned embeddings | Pre-trained representations |
| **Model** | Logistic regression, SVM, NB | RNN, CNN, Transformer | GPT, Claude, Llama |
| **Training data** | Hundreds–thousands | Thousands–millions | Fine-tune: hundreds; prompt: 0 |
| **Inference cost** | Milliseconds, CPU | Seconds, GPU | Seconds–minutes, GPU/API |
| **Interpretability** | High (feature weights) | Low | Very low |

**Decision framework:**
- Start with **TF-IDF + logistic regression** — fast baseline, interpretable, often sufficient.
- Move to **fine-tuned Transformers** when you need higher accuracy and have labeled data.
- Use **LLM prompting** for rapid prototyping, few-shot tasks, or when labeled data is scarce.
- Use **classical methods in production** when latency, cost, or interpretability constraints are tight.

---

## 3. Building a Complete Classical Pipeline

The following class encapsulates preprocessing, vectorization, training, prediction, and evaluation — the full workflow for a text classification task.

In [ ]:
class NLPPipeline:
    """End-to-end text classification pipeline."""

    def __init__(self, ngram_range=(1, 2), max_features=5000):
        self.pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(
                stop_words="english",
                ngram_range=ngram_range,
                max_features=max_features,
                min_df=1,
            )),
            ("clf", LogisticRegression(max_iter=1000, random_state=42)),
        ])
        self._trained = False

    @staticmethod
    def preprocess(text):
        text = re.sub(r"https?://\S+", "", text)
        text = re.sub(r"<[^>]+>", "", text)
        text = re.sub(r"[^\w\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip().lower()
        return text

    def fit(self, texts, labels):
        cleaned = [self.preprocess(t) for t in texts]
        self.pipeline.fit(cleaned, labels)
        self._trained = True
        return self

    def predict(self, texts):
        cleaned = [self.preprocess(t) for t in texts]
        return self.pipeline.predict(cleaned)

    def predict_proba(self, texts):
        cleaned = [self.preprocess(t) for t in texts]
        return self.pipeline.predict_proba(cleaned)

    def evaluate(self, texts, labels):
        preds = self.predict(texts)
        return {
            "accuracy": accuracy_score(labels, preds),
            "report": classification_report(labels, preds),
        }

    def explain(self, text, top_n=5):
        """Show top features driving the prediction."""
        tfidf = self.pipeline.named_steps["tfidf"]
        clf = self.pipeline.named_steps["clf"]
        cleaned = self.preprocess(text)
        vec = tfidf.transform([cleaned])
        features = tfidf.get_feature_names_out()
        coefs = clf.coef_[0]
        nonzero = vec.nonzero()[1]
        contributions = [(features[i], vec[0, i] * coefs[i]) for i in nonzero]
        contributions.sort(key=lambda x: -abs(x[1]))
        return contributions[:top_n]

In [ ]:
# Customer feedback classification dataset
feedback = [
    ("love this product works perfectly fast shipping", "positive"),
    ("amazing quality exceeded my expectations", "positive"),
    ("great customer service very helpful team", "positive"),
    ("excellent value for money highly recommend", "positive"),
    ("best purchase this year very satisfied", "positive"),
    ("wonderful experience will buy again", "positive"),
    ("terrible product broke after one day", "negative"),
    ("awful quality complete waste of money", "negative"),
    ("worst customer service ever very rude", "negative"),
    ("do not buy this product defective", "negative"),
    ("horrible experience want a refund", "negative"),
    ("disappointed with the quality poor design", "negative"),
    ("product is okay nothing special", "neutral"),
    ("average quality does the job", "neutral"),
    ("neither good nor bad just fine", "neutral"),
    ("acceptable but expected better", "neutral"),
    ("decent product for the price", "neutral"),
    ("works as described nothing more", "neutral"),
]

texts = [f[0] for f in feedback]
labels = [f[1] for f in feedback]

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels
)

nlp = NLPPipeline(ngram_range=(1, 2))
nlp.fit(X_train, y_train)
results = nlp.evaluate(X_test, y_test)

print(f"Test accuracy: {results['accuracy']:.3f}")
print(f"\n{results['report']}")

In [ ]:
# Predict on new reviews with confidence scores
new_reviews = [
    "fantastic product love it",
    "broken on arrival very upset",
    "it works fine i guess",
    "https://spam.com BUY NOW amazing deal!!!",
]

print(f"{'Review':<45s} {'Label':>8s} {'Confidence':>10s}")
print("-" * 65)
for review in new_reviews:
    pred = nlp.predict([review])[0]
    proba = nlp.predict_proba([review])[0]
    conf = proba.max()
    print(f"{review[:43]:<45s} {pred:>8s} {conf:10.3f}")

print("\nExplanation for 'fantastic product love it':")
for feat, contrib in nlp.explain("fantastic product love it"):
    print(f"  {feat:20s} {contrib:+.4f}")

---

## 4. Named Entity Recognition

**Named Entity Recognition (NER)** identifies and classifies entities in text — people, organizations, locations, dates, products.

*"Apple CEO Tim Cook announced a new iPhone in Cupertino on September 12."*

| Entity | Type |
|--------|------|
| Apple | ORGANIZATION |
| Tim Cook | PERSON |
| iPhone | PRODUCT |
| Cupertino | LOCATION |
| September 12 | DATE |

NER powers information extraction, knowledge graph construction, document indexing, and compliance (detecting PII in text). Modern systems use BiLSTM-CRF or Transformer models; rule-based patterns remain useful for structured formats.

In [ ]:
def simple_ner(text):
    """Rule-based NER for common entity patterns."""
    entities = []

    # Dates
    for m in re.finditer(r"\b(?:January|February|March|April|May|June|July|August|"
                         r"September|October|November|December)\s+\d{1,2}", text):
        entities.append({"text": m.group(), "type": "DATE", "start": m.start()})

    # Money
    for m in re.finditer(r"\$[\d,]+(?:\.\d{2})?", text):
        entities.append({"text": m.group(), "type": "MONEY", "start": m.start()})

    # Capitalized multi-word names (simple PERSON/ORG heuristic)
    for m in re.finditer(r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)+", text):
        entities.append({"text": m.group(), "type": "NAME", "start": m.start()})

    # Email addresses
    for m in re.finditer(r"\S+@\S+\.\S+", text):
        entities.append({"text": m.group(), "type": "EMAIL", "start": m.start()})

    entities.sort(key=lambda e: e["start"])
    return entities

documents = [
    "Tim Cook announced that Apple will invest $5,000,000 in Cupertino on September 12.",
    "Contact Jane Smith at jane.smith@company.com for details.",
    "Microsoft CEO Satya Nadella spoke at the conference in Seattle.",
]

for doc in documents:
    print(f"Text: {doc}")
    ents = simple_ner(doc)
    for e in ents:
        print(f"  [{e['type']:8s}] {e['text']}")
    print()

---

## 5. Question Answering

**Question answering (QA)** systems receive a question and a context document, then extract or generate the answer.

**Extractive QA** — find the answer span within the context (BERT-based models).

**Abstractive QA** — generate an answer that may paraphrase the context (seq2seq, LLMs).

**Open-domain QA** — retrieve relevant documents from a corpus, then answer (RAG architecture).

Modern QA pipelines: retrieve passages → rank by relevance → extract or generate answer → verify faithfulness.

In [ ]:
def extractive_qa(context, question, top_n=3):
    """Simple keyword-overlap QA baseline."""
    sentences = re.split(r"(?<=[.!?])\s+", context.strip())
    q_words = set(re.findall(r"\b[a-z]+\b", question.lower()))
    q_words -= {"what", "who", "where", "when", "how", "is", "are", "the", "a", "an", "did", "does"}

    scored = []
    for sent in sentences:
        s_words = set(re.findall(r"\b[a-z]+\b", sent.lower()))
        overlap = len(q_words & s_words)
        scored.append((sent, overlap))

    scored.sort(key=lambda x: -x[1])
    return scored[:top_n]

context = (
    "Python was created by Guido van Rossum and first released in 1991. "
    "It emphasizes code readability with significant indentation. "
    "Python supports multiple programming paradigms including procedural, "
    "object-oriented, and functional programming. "
    "It is widely used in data science, web development, and artificial intelligence."
)

questions = [
    "Who created Python?",
    "When was Python released?",
    "What is Python used for?",
]

for q in questions:
    answers = extractive_qa(context, q, top_n=1)
    print(f"Q: {q}")
    print(f"A: {answers[0][0]}\n")

---

## 6. Text Summarization

**Summarization** condenses a long document into a shorter version preserving key information.

**Extractive summarization** — select the most important existing sentences from the document. Simple approaches rank sentences by TF-IDF similarity to the document centroid or by position (lead sentences in news articles are often summaries).

**Abstractive summarization** — generate new sentences that paraphrase the content. Requires seq2seq models or LLMs. Produces more fluent summaries but risks hallucination.

Production systems often combine both: extract key sentences, then use an LLM to rewrite them concisely.

In [ ]:
def extractive_summary(text, num_sentences=2):
    """Rank sentences by word frequency importance."""
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    if len(sentences) <= num_sentences:
        return text

    word_freq = Counter()
    for sent in sentences:
        for w in re.findall(r"\b[a-z]+\b", sent.lower()):
            if len(w) > 3:
                word_freq[w] += 1

    scored = []
    for i, sent in enumerate(sentences):
        words = re.findall(r"\b[a-z]+\b", sent.lower())
        score = sum(word_freq[w] for w in words) / (len(words) + 1)
        scored.append((i, sent, score))

    top = sorted(scored, key=lambda x: -x[2])[:num_sentences]
    top.sort(key=lambda x: x[0])  # restore original order
    return " ".join(s for _, s, _ in top)

article = (
    "Artificial intelligence has transformed industries worldwide. "
    "Machine learning models now power recommendation systems, medical diagnosis, "
    "and autonomous vehicles. Deep learning, a subset of machine learning, uses neural "
    "networks with many layers to learn complex patterns. Large language models like GPT "
    "have demonstrated remarkable capabilities in text generation and understanding. "
    "However, challenges remain including bias, hallucination, and environmental impact. "
    "Researchers continue to develop safer and more efficient AI systems."
)

summary = extractive_summary(article, num_sentences=2)
print("Original article:")
print(f"  {article}\n")
print(f"Extractive summary ({len(summary.split())} words vs {len(article.split())} words):")
print(f"  {summary}")

---

## 7. Machine Translation

**Machine translation (MT)** converts text from one language to another. Modern systems use encoder-decoder Transformers trained on parallel corpora (millions of sentence pairs).

Historical progression:
- **Rule-based** — hand-written grammar and dictionary rules.
- **Statistical MT** — phrase tables from aligned bilingual text.
- **Neural MT** — seq2seq with attention (Google Translate, 2016).
- **Transformer MT** — current state of the art; supports 100+ languages.

Evaluation uses **BLEU** (bilingual evaluation understudy) — measures n-gram overlap between machine translation and human reference translations.

---

## 8. Information Retrieval and Search

**Information retrieval** finds documents relevant to a user query. The classic pipeline:

1. **Index** documents as TF-IDF vectors (or dense embeddings).
2. **Query** is vectorized with the same representation.
3. **Rank** documents by cosine similarity to the query.
4. **Return** top-k results.

Modern search combines sparse retrieval (BM25, an improved TF-IDF variant) with **dense retrieval** (embedding similarity) in hybrid systems. Google, Elasticsearch, and enterprise search engines use multi-stage retrieval: fast candidate generation → precise reranking.

In [ ]:
documents = [
    "Python is a versatile programming language for data science",
    "Machine learning algorithms learn patterns from data",
    "Neural networks are inspired by the human brain",
    "Natural language processing handles text and speech",
    "Web development with JavaScript and React frameworks",
    "Deep learning uses multiple layers of neural networks",
    "SQL databases store structured relational data",
    "Cloud computing provides scalable infrastructure",
]

vectorizer = TfidfVectorizer(stop_words="english")
doc_matrix = vectorizer.fit_transform(documents)

queries = ["python data science", "neural networks deep learning", "web javascript"]

for query in queries:
    q_vec = vectorizer.transform([query])
    scores = (doc_matrix @ q_vec.T).toarray().flatten()
    ranked = np.argsort(scores)[::-1][:3]
    print(f"Query: '{query}'")
    for idx in ranked:
        if scores[idx] > 0:
            print(f"  [{scores[idx]:.3f}] {documents[idx]}")
    print()

---

## 9. Chatbots and Dialogue Systems

Chatbots range from simple rule-based systems to sophisticated LLM-powered assistants.

| Type | How It Works | Example |
|------|-------------|--------|
| **Rule-based** | Pattern matching on keywords | "hours" → business hours response |
| **Retrieval-based** | Select best response from a database | FAQ bots |
| **Generative** | Language model generates responses | ChatGPT, Claude |
| **Task-oriented** | Intent detection + slot filling + action | "Book a flight to Paris on Friday" |

Task-oriented dialogue systems use a pipeline: **intent classification** (what does the user want?) → **entity extraction** (fill slots: destination, date) → **dialogue state tracking** → **action** (query API, confirm booking) → **response generation**.

In [ ]:
INTENTS = {
    "greeting": ["hello", "hi", "hey", "good morning"],
    "goodbye": ["bye", "goodbye", "see you", "thanks"],
    "hours": ["hours", "open", "close", "when are you"],
    "pricing": ["price", "cost", "how much", "pricing"],
    "support": ["help", "problem", "issue", "broken", "not working"],
}

RESPONSES = {
    "greeting": "Hello! How can I help you today?",
    "goodbye": "Goodbye! Have a great day.",
    "hours": "We're open Monday–Friday, 9 AM to 6 PM.",
    "pricing": "Our plans start at $9.99/month. Visit our pricing page for details.",
    "support": "I'm sorry to hear that. Let me connect you with our support team.",
    "fallback": "I'm not sure I understand. Could you rephrase that?",
}

def detect_intent(text):
    text_lower = text.lower()
    for intent, keywords in INTENTS.items():
        if any(kw in text_lower for kw in keywords):
            return intent
    return "fallback"

def chatbot(user_input):
    intent = detect_intent(user_input)
    return intent, RESPONSES[intent]

test_messages = [
    "Hello there!",
    "What are your hours?",
    "How much does it cost?",
    "My account is not working",
    "What's the weather today?",
    "Thanks, bye!",
]

print(f"{'User':<35s} {'Intent':<12s} Response")
print("-" * 80)
for msg in test_messages:
    intent, response = chatbot(msg)
    print(f"{msg:<35s} {intent:<12s} {response}")

---

## 10. Combining Multiple NLP Tasks

Production systems often chain multiple NLP components. A customer support pipeline might:

In [ ]:
class CustomerSupportPipeline:
    """Multi-task NLP pipeline: sentiment + intent + entity extraction."""

    def __init__(self, sentiment_model):
        self.sentiment_model = sentiment_model

    def process(self, text):
        cleaned = NLPPipeline.preprocess(text)
        sentiment = self.sentiment_model.predict([text])[0]
        sentiment_conf = self.sentiment_model.predict_proba([text])[0].max()
        intent, response = chatbot(text)
        entities = simple_ner(text)

        priority = "high" if sentiment == "negative" else "normal"
        if intent == "support":
            priority = "high"

        return {
            "original_text": text,
            "cleaned_text": cleaned,
            "sentiment": sentiment,
            "sentiment_confidence": round(float(sentiment_conf), 3),
            "intent": intent,
            "suggested_response": response,
            "entities": entities,
            "priority": priority,
        }


support = CustomerSupportPipeline(nlp)

tickets = [
    "My order is broken and I want a refund immediately!",
    "What are your business hours?",
    "Love the new update, great work team!",
    "Contact Sarah Johnson at sarah@corp.com about the $500 invoice.",
]

for ticket in tickets:
    result = support.process(ticket)
    print(f"Ticket: {result['original_text']}")
    print(f"  Sentiment:  {result['sentiment']} ({result['sentiment_confidence']:.2f})")
    print(f"  Intent:     {result['intent']}")
    print(f"  Priority:   {result['priority']}")
    print(f"  Response:   {result['suggested_response']}")
    if result['entities']:
        print(f"  Entities:   {result['entities']}")
    print()

---

## 11. Multilingual NLP

NLP is not English-only. Challenges for multilingual systems:

- **Scripts** — Latin, Cyrillic, Arabic, Devanagari, CJK characters require appropriate tokenizers.
- **Morphology** — agglutinative languages (Turkish, Finnish) have rich word forms.
- **Word order** — SOV (Japanese, Hindi) vs SVO (English) affects parsing.
- **Resource availability** — less training data and fewer pre-trained models for low-resource languages.

Modern multilingual models (mBERT, XLM-RoBERTa, NLLB) are trained on 100+ languages and transfer knowledge across languages. For low-resource languages, **cross-lingual transfer** — fine-tuning a multilingual model with limited target-language data — is the standard approach.

---

## 12. Deploying NLP Systems

Moving from notebook to production requires attention to engineering concerns:

**Model serialization** — save the trained pipeline (scikit-learn `joblib`, ONNX, or model hub format) for consistent inference.

**API design** — expose predictions via REST or gRPC endpoints. Accept text input, return structured JSON.

```json
POST /classify
{"text": "Great product, love it!"}

Response:
{"label": "positive", "confidence": 0.94, "latency_ms": 12}
```

**Batch vs real-time** — batch processing for offline analysis (overnight document classification); real-time for chatbots and autocomplete.

**Latency** — classical models run in milliseconds on CPU. Transformer models need GPU for acceptable latency. LLM APIs add network overhead.

**Versioning** — track model versions, training data, and hyperparameters. Roll back if a new model performs worse in production.

In [ ]:
def serve_prediction(pipeline, text):
    """Simulate a production API response."""
    import time
    start = time.perf_counter()
    label = pipeline.predict([text])[0]
    proba = pipeline.predict_proba([text])[0]
    latency_ms = (time.perf_counter() - start) * 1000

    return json.dumps({
        "text": text,
        "label": label,
        "confidence": round(float(proba.max()), 4),
        "probabilities": {
            cls: round(float(p), 4)
            for cls, p in zip(pipeline.pipeline.classes_, proba)
        },
        "latency_ms": round(latency_ms, 2),
        "model_version": "tfidf-lr-v1.0",
    }, indent=2)

print(serve_prediction(nlp, "absolutely love this product"))

---

## 13. Monitoring and Maintenance

Models degrade over time as language, user behavior, and data distributions shift.

**What to monitor:**
- **Prediction distribution** — sudden shift in class proportions may indicate data drift.
- **Confidence scores** — declining average confidence suggests the model is uncertain on new data.
- **Latency** — inference time per request.
- **Error rate** — user corrections, negative feedback, manual review findings.
- **Input characteristics** — average text length, language, vocabulary overlap with training data.

**When to retrain:**
- Performance drops below an acceptable threshold on a validation set sampled from production data.
- New categories or vocabulary emerge (new product names, slang, domain terminology).
- Scheduled periodic retraining (monthly/quarterly) with accumulated labeled data.

In [ ]:
# Simulated production monitoring dashboard
np.random.seed(42)
days = np.arange(30)
accuracy = 0.92 - 0.002 * days + np.random.randn(30) * 0.01
latency = 15 + 0.3 * days + np.random.randn(30) * 2
volume = 1000 + 50 * days + np.random.randn(30) * 100

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(days, accuracy, "b-o", markersize=3)
axes[0].axhline(0.85, color="red", linestyle="--", alpha=0.5, label="Retrain threshold")
axes[0].set_xlabel("Day")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Model accuracy over time")
axes[0].legend()

axes[1].plot(days, latency, "g-o", markersize=3)
axes[1].set_xlabel("Day")
axes[1].set_ylabel("Latency (ms)")
axes[1].set_title("Inference latency")

axes[2].bar(days[::3], volume[::3], color="coral", width=2)
axes[2].set_xlabel("Day")
axes[2].set_ylabel("Requests/day")
axes[2].set_title("Request volume")

plt.suptitle("Production monitoring dashboard", y=1.02)
plt.tight_layout()
plt.show()

---

## 14. Responsible NLP

NLP systems interact with human language and affect real people. Responsible development requires:

**Bias detection and mitigation** — models trained on web text inherit societal biases. Test performance across demographic groups. Use balanced training data and fairness-aware evaluation.

**Privacy** — NLP systems process personal communications, medical records, and financial documents. Implement data anonymization, comply with regulations (GDPR, HIPAA), and minimize data retention.

**Transparency** — users should know when they interact with an AI system. Disclose automated decision-making in high-stakes domains (hiring, lending, healthcare).

**Hallucination management** — generative models produce plausible but false information. Use RAG for factual queries, cite sources, and add human review for critical applications.

**Environmental impact** — training large models consumes significant energy. Use efficient architectures, smaller task-specific models, and cloud providers with renewable energy commitments when possible.

---

## 15. The Modern NLP Stack

A practitioner's toolkit for building NLP systems today:

| Layer | Tools |
|-------|-------|
| **Data** | pandas, Hugging Face Datasets, Label Studio |
| **Preprocessing** | spaCy, NLTK, Hugging Face Tokenizers |
| **Classical ML** | scikit-learn (TF-IDF + classifiers) |
| **Neural models** | PyTorch, Hugging Face Transformers |
| **LLM APIs** | OpenAI, Anthropic, local Llama/Ollama |
| **Vector search** | FAISS, Pinecone, ChromaDB |
| **Orchestration** | LangChain, LlamaIndex |
| **Deployment** | FastAPI, Docker, ONNX Runtime |
| **Monitoring** | Weights & Biases, MLflow, Prometheus |

Start simple. A TF-IDF classifier in scikit-learn solves many real problems. Add complexity — embeddings, Transformers, LLMs — only when the baseline is clearly insufficient and the business case justifies the additional cost.

---

## 16. Capstone Workflow Checklist

When building any NLP system, follow this checklist:

1. **Define the task** — classification, extraction, generation, or search?
2. **Collect and explore data** — size, class balance, language, noise level.
3. **Establish a baseline** — TF-IDF + logistic regression with cross-validation.
4. **Choose metrics** — accuracy, F1, BLEU, perplexity, or human evaluation.
5. **Build preprocessing pipeline** — clean, tokenize, vectorize consistently.
6. **Train and evaluate** — stratified splits, confusion matrix, error analysis.
7. **Iterate** — n-grams, class weights, more data, or upgrade to neural models.
8. **Package for deployment** — serialize pipeline, build API, set up monitoring.
9. **Test in production** — shadow mode first, then gradual rollout.
10. **Monitor and retrain** — track drift, collect feedback, improve continuously.

---

## 17. Summary

NLP systems combine preprocessing, representation, modeling, and deployment into pipelines that solve real problems — from classifying customer feedback to powering conversational assistants.

The **classical pipeline** (clean → TF-IDF → classifier) remains a strong, interpretable baseline. **Neural models** and **LLMs** extend capabilities to generation, complex understanding, and few-shot learning — at higher computational cost.

Major applications include **text classification**, **named entity recognition**, **question answering**, **summarization**, **machine translation**, **information retrieval**, and **dialogue systems**. Production systems chain multiple NLP tasks and require careful attention to **deployment**, **monitoring**, and **responsible use**.

The field has evolved from hand-written rules to statistical models to neural networks to billion-parameter language models — but the fundamental workflow remains: represent text as numbers, learn patterns from data, evaluate rigorously, and deploy responsibly.